In [ ]:
!pip -q install rdkit==2026.03.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 51.5 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# Load the CSV
df = pd.read_csv("ANPDB_fungi_with_bioactivity.csv")
# Preview
print(df.columns)

Index(['molecule_id', 'mol_name', 'pubchem_id', 'smiles', 'inchi',
       'source_species', 'family', 'kingdom', 'class_name', 'subclass_name',
       'synonyms', 'pmids', 'links', 'bioactivity'],
      dtype='object')


In [ ]:
bio_col = "bioactivity"  # column with comma-separated activities

# Split to clean lists
df["bio_list"] = (
    df[bio_col].fillna("").astype(str)
      .str.split(",")
      .apply(lambda xs: [x.strip() for x in xs if x.strip() != ""])
)

# Collect all unique labels
all_labels = sorted({lab for lst in df["bio_list"] for lab in lst})

# Create one-hot columns
for lab in all_labels:
    df[lab] = df["bio_list"].apply(lambda lst: int(lab in lst))

In [ ]:
df

,molecule_id,mol_name,pubchem_id,smiles,inchi,source_species,family,kingdom,class_name,subclass_name,...,Inhibitory toward p56lck tyrosine,Kinase inhibitory,Motility-impairing,Nematicidal,Phytotoxicity,Protein kinase inhibition,Suppressive on phagocytosis response upon activation with serum,Toxin,Vasodepressor,Zootoxic
0,Mol_02704,(-)-dihydroepiepoformin,10419279.0,O=C1[C@H](C)C[C@@H]([C@@H]2[C@H]1O2)O,InChI=1S/C7H10O3/c1-3-2-4(8)6-7(10-6)5(3)9/h3-...,Geosmithia langdonii,Incertae sedis,Fungi,epiepoformin,NaN,...,0,0,0,0,0,0,0,0,0,0
1,Mol_00945,(+)-epiepoformin,155055.0,O[C@H]1C=C(C)C(=O)[C@H]2[C@@H]1O2,"InChI=1S/C7H8O3/c1-3-2-4(8)6-7(10-6)5(3)9/h2,4...",Diplodia quercivora || Geosmithia langdonii,Botryosphaeriaceae || Incertae sedis,Fungi,terpenoid,pimarane diterpenoid,...,0,0,0,0,1,0,0,0,0,1
2,Mol_01253,(+)-sclerotiorin,6436015.0,CC[C@@H](/C=C(/C=C/C1=CC2=C(Cl)C(=O)[C@](C(=O)...,InChI=1S/C21H23ClO5/c1-6-12(2)9-13(3)7-8-15-10...,Chaetomium aureum,Chaetomiaceae,Fungi,azaphilone,NaN,...,0,0,0,0,0,0,0,0,0,0
3,Mol_02702,"(4R,5R,6R)-4,5-dihydroxy-6-(6'-methylsalicylox...",162817181.0,O[C@@H]1C=C(C)C(=O)[C@@H]([C@@H]1O)OCc1c(C)cccc1O,InChI=1S/C15H18O5/c1-8-4-3-5-11(16)10(8)7-20-1...,Geosmithia langdonii,Incertae sedis,Fungi,ketone,NaN,...,0,0,0,0,0,0,0,0,0,0
4,Mol_02705,"(4S,5S)-4,5-dihydroxy-2-methylcyclohex-2-enone",70677551.0,O[C@H]1C=C(C)C(=O)C[C@@H]1O,"InChI=1S/C7H10O3/c1-4-2-6(9)7(10)3-5(4)8/h2,6-...",Geosmithia langdonii,Incertae sedis,Fungi,ketone,NaN,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,Mol_04101,trans-dehydrocurvularin,6438143.0,C[C@H]1CCC/C=C/C(=O)c2c(CC(=O)O1)cc(O)cc2O,InChI=1S/C16H18O5/c1-10-5-3-2-4-6-13(18)16-11(...,Penicillium species,Trichocomaceae,Fungi,lactone,macrolide,...,0,0,0,0,0,0,0,0,0,0
171,Mol_04111,trichodimerol,21778326.0,C/C=C/C=C/C(=C\1/C(=O)[C@@]2(C)[C@H]3/C(=C(\C=...,InChI=1S/C28H32O8/c1-7-9-11-13-15(29)17-19-23(...,Penicillium species,Trichocomaceae,Fungi,heterocyclic,heterotetracyclic,...,0,0,0,0,0,0,0,0,0,0
172,Mol_04076,tryprostatin B,9863281.0,CC(=CCc1[nH]c2c(c1C[C@@H]1NC(=O)[C@H]3N(C1=O)C...,InChI=1S/C21H25N3O2/c1-13(2)9-10-17-15(14-6-3-...,Aspergillus fumigatus,Trichocomaceae,Fungi,alkaloid,diketopiperazine,...,0,0,0,0,0,0,0,0,0,0
173,Mol_02555,violaceol I,100615.0,Cc1cc(Oc2cc(C)cc(c2O)O)c(c(c1)O)O,InChI=1S/C14H14O5/c1-7-3-9(15)13(17)11(5-7)19-...,Aspergillus austroafricanus || Aspergillus ver...,Eurotiomycetidae || Trichocomaceae,Fungi,phenolic,phenolic ether,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.columns

Index(['molecule_id', 'mol_name', 'pubchem_id', 'smiles', 'inchi',
       'source_species', 'family', 'kingdom', 'class_name', 'subclass_name',
       'synonyms', 'pmids', 'links', 'bioactivity', 'bio_list', 'Anthelmintic',
       'Anti-inflammatory', 'Anti-leishmanial', 'Anti-plasmodial',
       'Anti-respiratory syncytial virus (rsv)', 'Antibacterial', 'Anticancer',
       'Anticholinesterase', 'Antifungal', 'Antileukemia', 'Antimicrobial',
       'Antimigratory', 'Antioxidant', 'Antiproliferative', 'Antitumor',
       'Cell proliferation inhibition', 'Cytotoxic', 'Hdac inhibition',
       'Inhibition of the hsp90 machine chaperoning',
       'Inhibitory toward p56lck tyrosine', 'Kinase inhibitory',
       'Motility-impairing', 'Nematicidal', 'Phytotoxicity',
       'Protein kinase inhibition',
       'Suppressive on phagocytosis response upon activation with serum',
       'Toxin', 'Vasodepressor', 'Zootoxic'],
      dtype='object')

In [ ]:
# Count bioactivities
bio_cols = ['Anthelmintic', 'Anti-inflammatory', 'Anti-leishmanial',
       'Anti-plasmodial', 'Anti-respiratory syncytial virus (rsv)',
       'Antibacterial', 'Anticancer', 'Anticholinesterase', 'Antifungal',
       'Antileukemia', 'Antimicrobial', 'Antimigratory', 'Antioxidant',
       'Antiproliferative', 'Antitumor', 'Cell proliferation inhibition',
       'Cytotoxic', 'Hdac inhibition',
       'Inhibition of the hsp90 machine chaperoning',
       'Inhibitory toward p56lck tyrosine', 'Kinase inhibitory',
       'Motility-impairing', 'Nematicidal', 'Phytotoxicity',
       'Protein kinase inhibition',
       'Suppressive on phagocytosis response upon activation with serum',
       'Toxin', 'Vasodepressor', 'Zootoxic']
print("\nCounts per bioactivity:\n")
for col in bio_cols:
    print(f"{col} -> {int(df[col].sum())} compounds")


Counts per bioactivity:

Anthelmintic -> 3 compounds
Anti-inflammatory -> 1 compounds
Anti-leishmanial -> 15 compounds
Anti-plasmodial -> 4 compounds
Anti-respiratory syncytial virus (rsv) -> 1 compounds
Antibacterial -> 36 compounds
Anticancer -> 8 compounds
Anticholinesterase -> 1 compounds
Antifungal -> 15 compounds
Antileukemia -> 6 compounds
Antimicrobial -> 34 compounds
Antimigratory -> 2 compounds
Antioxidant -> 2 compounds
Antiproliferative -> 2 compounds
Antitumor -> 5 compounds
Cell proliferation inhibition -> 10 compounds
Cytotoxic -> 49 compounds
Hdac inhibition -> 10 compounds
Inhibition of the hsp90 machine chaperoning -> 1 compounds
Inhibitory toward p56lck tyrosine -> 1 compounds
Kinase inhibitory -> 8 compounds
Motility-impairing -> 4 compounds
Nematicidal -> 2 compounds
Phytotoxicity -> 4 compounds
Protein kinase inhibition -> 10 compounds
Suppressive on phagocytosis response upon activation with serum -> 1 compounds
Toxin -> 3 compounds
Vasodepressor -> 1 compounds


In [ ]:
# Remove rows where SMILES is missing in meta file
before = len(df)
df = df[df["smiles"].notna()].reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} compounds with no SMILES.")
print(f"Final dataset size: {after} compounds")

Removed 0 compounds with no SMILES.
Final dataset size: 175 compounds


In [ ]:
# Function: SMILES -> canonical SMILES
def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except:
        return None  # invalid SMILES

In [ ]:
# Function: SMILES -> InChIKey
def smiles_to_inchikey(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return None
        return inchi.MolToInchiKey(mol)
    except:
        return None

In [ ]:
# Compute canonical smiles
df["canonical_smiles"] = df["smiles"].apply(canonicalize_smiles)
# Report how many succeeded
success_CS = df["canonical_smiles"].notna().sum()
total = len(df)
print(f"Generated canonical smiles for {success_CS}/{total} compounds ({success_CS/total*100:.1f}%).")

Generated canonical smiles for 175/175 compounds (100.0%).


In [ ]:
# Compute InChIKeys
df["InChIKey"] = df["smiles"].apply(smiles_to_inchikey)
# Report how many succeeded
success_InChI = df["InChIKey"].notna().sum()
total = len(df)
print(f"Generated InChIKeys for {success_InChI}/{total} compounds ({success_InChI/total*100:.1f}%).")

Generated InChIKeys for 175/175 compounds (100.0%).


In [ ]:
df.columns

Index(['molecule_id', 'mol_name', 'pubchem_id', 'smiles', 'inchi',
       'source_species', 'family', 'kingdom', 'class_name', 'subclass_name',
       'synonyms', 'pmids', 'links', 'bioactivity', 'bio_list', 'Anthelmintic',
       'Anti-inflammatory', 'Anti-leishmanial', 'Anti-plasmodial',
       'Anti-respiratory syncytial virus (rsv)', 'Antibacterial', 'Anticancer',
       'Anticholinesterase', 'Antifungal', 'Antileukemia', 'Antimicrobial',
       'Antimigratory', 'Antioxidant', 'Antiproliferative', 'Antitumor',
       'Cell proliferation inhibition', 'Cytotoxic', 'Hdac inhibition',
       'Inhibition of the hsp90 machine chaperoning',
       'Inhibitory toward p56lck tyrosine', 'Kinase inhibitory',
       'Motility-impairing', 'Nematicidal', 'Phytotoxicity',
       'Protein kinase inhibition',
       'Suppressive on phagocytosis response upon activation with serum',
       'Toxin', 'Vasodepressor', 'Zootoxic', 'canonical_smiles', 'InChIKey'],
      dtype='object')

In [ ]:
df_out = df[['molecule_id', 'mol_name', 'pubchem_id', 'smiles',
       'source_species', 'pmids', 'links', 'bio_list', 'Anthelmintic',
       'Anti-inflammatory', 'Anti-leishmanial', 'Anti-plasmodial',
       'Anti-respiratory syncytial virus (rsv)', 'Antibacterial', 'Anticancer',
       'Anticholinesterase', 'Antifungal', 'Antileukemia', 'Antimicrobial',
       'Antimigratory', 'Antioxidant', 'Antiproliferative', 'Antitumor',
       'Cell proliferation inhibition', 'Cytotoxic', 'Hdac inhibition',
       'Inhibition of the hsp90 machine chaperoning',
       'Inhibitory toward p56lck tyrosine', 'Kinase inhibitory',
       'Motility-impairing', 'Nematicidal', 'Phytotoxicity',
       'Protein kinase inhibition',
       'Suppressive on phagocytosis response upon activation with serum',
       'Toxin', 'Vasodepressor', 'Zootoxic', 'canonical_smiles', 'InChIKey']]

In [ ]:
df_out.shape

(175, 39)

In [ ]:
df_out.to_csv("ANPDB_processed.csv", index=False)